# 🎬 AI Movie Recommender — Complete Pipeline

A hybrid AI recommendation system built on three composable layers:

| Layer | Technology | Role |
|---|---|---|
| **1 — Collaborative Filtering** | SVD via `scikit-surprise` | Predicts per-user ratings for every unseen movie |
| **2 — Semantic Search / RAG** | `sentence-transformers` + FAISS | Retrieves contextually relevant movies for LLM grounding |
| **3 — LLM Explanations** | OpenAI GPT-3.5 or Groq LLaMA-3 | Generates natural-language justifications tied to watch history |

**Datasets:** MovieLens 1M (auto-download) · TMDB 5000 (Kaggle / manual upload)  
**Run every cell in order from top to bottom.**

In [ ]:
# Clone the repo
!git clone https://github.com/RahulKakani9999/AI-Powered-Netflix-Movie-Recommendation-System.git
%cd AI-Powered-Netflix-Movie-Recommendation-System

# Install dependencies not pre-installed on Colab
!pip install -q scikit-surprise sentence-transformers faiss-cpu openai groq streamlit python-dotenv
print("\u2713 Dependencies installed")

## Step 1: Configure API Key

Choose **one** of the two options below.

**Option A — Colab Secrets (recommended, key stays private):**
1. Click the **🔑 Secrets** icon in the left sidebar
2. Add a new secret: `GROQ_API_KEY` → your key from [console.groq.com](https://console.groq.com) *(free)*
3. Run the cell — it will load automatically

**Option B — Hardcoded (quick but insecure):** paste your key directly in the cell.  
> ⚠️ Delete the key before sharing or saving the notebook.

In [ ]:
import os

# Option A: Colab Secrets (recommended)
try:
    from google.colab import userdata
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
    print("\u2713 API key loaded from Colab Secrets")
except Exception:
    # Option B: Paste your key here (delete before sharing!)
    os.environ['GROQ_API_KEY'] = 'your-groq-api-key-here'
    print("\u26a0 Using hardcoded key \u2014 remove before sharing!")

os.environ['LLM_PROVIDER'] = 'groq'
print(f"Provider : {os.environ['LLM_PROVIDER']}")
print(f"Key set  : {'Yes' if os.environ.get('GROQ_API_KEY') else 'No'}")

## Step 2: Download Datasets

### Part A — MovieLens 1M
~25 MB, downloaded and extracted automatically by the script below.

### Part B — TMDB 5000
Requires a free Kaggle account. Two options:
- **Kaggle API** *(faster)*: uncomment the Kaggle block in the next cell and upload your `kaggle.json`
- **Manual upload** *(no setup needed)*: run the cell and upload the two CSV files when prompted

Download link: [kaggle.com/datasets/tmdb/tmdb-movie-metadata](https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata)

In [ ]:
# Download MovieLens 1M (~25 MB, auto-extracted to data/movielens/)
!python data/download_data.py

### Download TMDB 5000 from Kaggle

Run the cell below and choose **one** method:
- **Uncomment the Kaggle block** if you have `kaggle.json` (upload it first via the Files panel)
- **Leave the upload block active** to use the Colab file-upload dialog

In [ ]:
# ── Option A: Kaggle API ──────────────────────────────────────────────────
# 1. Download kaggle.json from Kaggle → Account → Create New Token
# 2. Upload kaggle.json via the Files panel (left sidebar)
# 3. Uncomment and run the four lines below:
#
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !pip install -q kaggle
# !kaggle datasets download -d tmdb/tmdb-movie-metadata -p data/tmdb/ --unzip
# print("\u2713 TMDB dataset downloaded via Kaggle API")

# ── Option B: Manual file upload ─────────────────────────────────────────
from google.colab import files
import os

os.makedirs('data/tmdb', exist_ok=True)
print("Upload tmdb_5000_movies.csv and tmdb_5000_credits.csv:")
uploaded = files.upload()
for filename, content in uploaded.items():
    dest = f'data/tmdb/{filename}'
    with open(dest, 'wb') as f:
        f.write(content)
    print(f"  \u2713 Saved {filename} ({len(content):,} bytes)")

## Step 3: Preprocess Data

Merges MovieLens and TMDB, extracts year + `clean_title`, parses JSON metadata columns, and writes three files to `data/processed/`:

| File | Contents |
|---|---|
| `ratings.csv` | `user_id`, `movie_id`, `rating`, `timestamp` |
| `movies.csv` | Full merged dataset |
| `movies_metadata.csv` | Slim version with `combined_text` for the FAISS index |

In [ ]:
!python data/preprocess.py

## Step 4: Train the SVD Model

Trains Singular Value Decomposition on all 1 M ratings:
1. **5-fold cross-validation** — prints RMSE ± std and MAE ± std per fold
2. **Full training** on `build_full_trainset()` — saves `svd_model.pkl` + `trainset.pkl`

> ⏱ *Approximately 3–5 minutes on a free Colab GPU/CPU.*

In [ ]:
!python models/train_svd.py

## Step 5: Build the FAISS Vector Index

Encodes each movie's `combined_text` (title + overview + genres + keywords + cast + director) using `all-MiniLM-L6-v2` and builds a FAISS `IndexFlatIP` for cosine similarity search.

Saves to `models/saved/`:
- `faiss_index.bin` — the vector index (one vector per movie)
- `indexed_movies.csv` — metadata rows aligned with index positions
- `embedding_model_name.txt` — model name used for query encoding

> ⏱ *First run downloads the ~90 MB embedding model, then encodes ~3 800 movies.*

In [ ]:
!python models/build_faiss_index.py

## Step 6: Evaluate the Model

Runs automated metrics on a synthetic 80/20 holdout split:

| Metric | Target | Description |
|---|---|---|
| **RMSE** | ~0.87 | Rating prediction error (lower = better) |
| **Precision@10** | ~0.73 | Fraction of top-10 recommendations with actual rating ≥ 4★ |

In [ ]:
!python evaluation/evaluate_model.py

## Step 7: Test the Full Pipeline Interactively

Loads all three layers via `RecommendationPipeline` and generates SVD recommendations with LLM explanations for a sample user.

In [ ]:
import sys
sys.path.insert(0, '.')
from utils.pipeline import RecommendationPipeline

# Initialise the pipeline (loads SVD model, FAISS index, LLM client)
pipeline = RecommendationPipeline()

# Get demo users with >= 50 ratings
valid_users = pipeline.get_valid_users(sample=5)
print(f"Sample user IDs with 50+ ratings: {valid_users}")

# Pick the first user for demo
user_id = valid_users[0]
print(f"\n{'='*60}")
print(f"Recommendations for User {user_id}")
print(f"{'='*60}")

# Get explained recommendations
result = pipeline.get_explained_recommendations(user_id, n=5)

# Display top-5 movies
print("\nTop 5 Recommended Movies:")
for i, (_, row) in enumerate(result['recommendations'].head(5).iterrows(), 1):
    title = row.get('title', 'Unknown')
    genres = row.get('genres_list', '')
    pred = row.get('predicted_rating', 0)
    print(f"  {i}. {title}  |  Predicted: {pred:.2f}/5")

# Display LLM explanation
print(f"\n{'='*60}")
print("LLM Explanation:")
print(f"{'='*60}")
print(result['explanation'])

## Step 8: Chat with the Recommender

Conversational interface powered by FAISS semantic search (RAG) + LLM.  
Type your question and press **Enter**. Type `quit` to stop the loop.

In [ ]:
print("\U0001f4ac Chat with the AI Movie Recommender")
print("Type 'quit' to exit\n")

conversation_history = []

while True:
    user_input = input("You: ")
    if user_input.lower() in ['quit', 'exit', 'q']:
        print("Goodbye! \U0001f3ac")
        break

    response = pipeline.chat(
        user_message=user_input,
        conversation_history=conversation_history
    )

    print(f"\nAssistant: {response}\n")

    conversation_history.append({"role": "user", "content": user_input})
    conversation_history.append({"role": "assistant", "content": response})

    # Keep history manageable
    if len(conversation_history) > 10:
        conversation_history = conversation_history[-10:]

## Step 9: Semantic Movie Search

Tests the FAISS vector search layer directly with natural-language queries — no SVD or LLM involved.  
Results are ranked by cosine similarity score (0 → 1, higher = closer).

In [ ]:
queries = [
    "dark thriller with plot twists like Inception",
    "funny romantic comedy for date night",
    "epic sci-fi adventure in space",
    "classic gangster crime movie",
]

for query in queries:
    print(f"\n\U0001f50d '{query}'")
    results = pipeline.search_movies(query, top_k=3)
    for _, row in results.iterrows():
        title = row.get('title', 'Unknown')
        score = row.get('similarity_score', 0)
        print(f"   \u2192 {title}  [score: {score:.3f}]")

## Step 10: Launch Streamlit UI *(Optional)*

Runs the full chat interface (`app.py`) on port 8501 and exposes it via a `localtunnel` public URL.

> **Notes:**
> - The tunnel URL is printed after ~10 seconds — click it to open the app
> - The URL changes every session
> - The cell runs indefinitely; use **Runtime → Interrupt execution** to stop it
> - If the tunnel prompt asks for a password, enter the external IP shown in the output

In [ ]:
# Install localtunnel (Node.js, pre-available on Colab)
!npm install -g localtunnel 2>/dev/null

import subprocess, threading, time

def run_streamlit():
    subprocess.Popen(
        ['streamlit', 'run', 'app.py',
         '--server.port', '8501',
         '--server.headless', 'true'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()
time.sleep(8)  # wait for Streamlit to start

print("Streamlit is running on port 8501. Starting tunnel ...")
!npx localtunnel --port 8501